In [1]:
from pathlib import Path
from mhdr.dataloader.io import read_csv, save_csv
INPUT_DIR = Path.cwd() / "input"
OUTPUT_DIR = Path.cwd() / "output"
TEMP_DIR = Path.cwd() / "temp"

In [2]:
from mhdr.generator.flan_t5_generator import FlanGenerator
import pandas as pd
from pathlib import Path

questions_df = read_csv(INPUT_DIR / "questions_master.csv")

def build_prompt(seed_text):
    return f"""
    Rewrite the following mental health statement in a different way.

    Keep the core meaning.

    Change wording, structure, and phrasing significantly.
    Use varied sentence forms.
    Do not copy phrases from the original.

    Output only one statement.

    Original:
    {seed_text}
    """

gen = FlanGenerator()

OUTPUT_PATH = OUTPUT_DIR / "generated_questions.csv"

batch_size = 20
first_write = True

for start in range(0, len(questions_df), batch_size):
    batch = questions_df.iloc[start:start+batch_size]

    rows = []

    for _, r in batch.iterrows():
        seed_text = r["text"]
        seed_qid = r["qid"]
        seed_dataset = r.get("dataset", "")

        for i in range(10):
            prompt = build_prompt(seed_text)

            out = gen.generate(
                prompt,
                num_return_sequences=1,
                max_new_tokens=30,
                temperature=0.8,
                top_p=0.9,
            )[0].strip()

            rows.append({
                "seed_qid": seed_qid,
                "seed_dataset": seed_dataset,
                "seed_text": seed_text,
                "text": out
            })

    batch_df = pd.DataFrame(rows)

    batch_df.to_csv(
        OUTPUT_PATH,
        mode="w" if first_write else "a",
        header=first_write,
        index=False,
    )

    first_write = False

    print(f"Processed {start + len(batch)} / {len(questions_df)}")

print("Done.")

/Users/haikeyu/Desktop/mentalhealth-dimension-reduction/.venv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 190/190 [00:00<00:00, 1921.06it/s, Materializing param=shared.weight]                                                       
The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning


Processed 20 / 145
Processed 40 / 145
Processed 60 / 145
Processed 80 / 145
Processed 100 / 145
Processed 120 / 145
Processed 140 / 145
Processed 145 / 145
Done.
